In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import numpy as np
import datasets

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [6]:
from tqdm.notebook import tqdm

In [7]:
import sys
sys.path.append('..')
from utils.reasoning import make_segment, split_cot
from torch.nn.utils.rnn import pad_sequence

In [8]:
from modeling_rmt.language_modeling import MemoryCell
from modeling_rmt.experimental import RecurrentWrapperNoSegmentationGenerate

In [9]:
device = 'cuda'
model_name = "HuggingFaceTB/SmolLM2-135M"
checkpoint_path = "/home/user33/kashurin/RMT_SmolLM2-135M/cot_fixed_pad/checkpoint-2700/pytorch_model.bin"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
bos = [tokenizer.bos_token_id]
eos = [tokenizer.eos_token_id]
think = tokenizer.encode("<issue_start>")
ans = tokenizer.encode("<issue_closed>")

delim = ">> <<"


memory_cell = MemoryCell(
    model,
    num_mem_tokens=16
)

model = RecurrentWrapperNoSegmentationGenerate(memory_cell, 
                                             max_n_segments=10, 
                                             think_token_id=think[0],
                                             answer_token_id=ans[0],
                                             bos_token_id=bos[0],
                                             eos_token_id=eos[0]
                                             )

model.load_state_dict(torch.load(checkpoint_path), strict=False)
model.to(device)
print(':)')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


:)


In [7]:
prompts = [
    "The future of AI is",
    "In a galaxy far far away",
    "Hello"
]

In [8]:
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer(prompts, return_tensors="pt", padding=True, padding_side='left')
inputs

{'input_ids': tensor([[    0,   504,  1774,   282,  5646,   314],
        [  788,   253, 13247,  1869,  1869,  2025],
        [    0,     0,     0,     0,     0, 19556]]), 'attention_mask': tensor([[0, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1]])}

In [10]:
class Holder:
    def __init__(self):
        pass
args = Holder()
args.num_mem_tokens = 16
args.task_name = 'gsm8k'
args.max_new_tokens = 100

In [11]:
def collate_fn(batch):
    # first, we segment each sample into task, cot steps and labels
    segments_batch = []
    for sample in batch:
        task, lab, cot = sample['task'], sample['labels'], sample['cot']
        task_tokens = tokenizer.encode(task, add_special_tokens=False)
        labels_tokens = tokenizer.encode(lab, add_special_tokens=False)
        cot_segments = split_cot(cot, by=delim)
        cot_segment_tokens = tokenizer.batch_encode_plus(cot_segments, add_special_tokens=False)['input_ids']

        segments = []
        segments.append(make_segment(bos + task_tokens + think, loss=False))
        for segment in cot_segment_tokens[:-1]:
            segments.append(make_segment(bos + segment + think, loss=True))
        segments.append(make_segment(bos + cot_segment_tokens[-1] + ans, loss=True))

        segments.append(make_segment(bos + labels_tokens + eos, loss=True))
        segments_batch.append(segments)

    # if some samples have less segments than others, we pad them with empty segments
    num_segments = max(len(segments) for segments in segments_batch)
    for segments in segments_batch:
        if len(segments) < num_segments:
            segments.extend([make_segment(eos, loss=False)] * (num_segments - len(segments)))

    # prepare segments for the whole batch
    batch_segments = []
    for i in range(num_segments):
        padding_side = 'right'
        if i == 0:
            padding_side = 'left'

        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad, padding_side=padding_side)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0, padding_side=padding_side)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100, padding_side=padding_side)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False, padding_side=padding_side)

        batch_segment = {'input_ids': input_ids,
                            'attention_mask': attention_mask,
                            'labels_mask': labels_mask,
                            'labels': labels
                            }
        
        batch_segments.append(batch_segment)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, 'labels': full_labels}

In [12]:
dataset = 'booydar/gsm8k'
train_dataset = datasets.load_dataset(dataset, split='train')
valid_dataset = datasets.load_dataset(dataset, split='valid')

In [8]:
I = 10

In [21]:
all_preds, all_labels = [], []
all_preds_cot, all_labels_cot = [], []
all_preds_ans, all_labels_ans = [], []

batch = valid_dataset.select(range(I, I+1))
collated = collate_fn(batch)
task = collated['segments'][0]
task = {k:v.to(device) for k,v in task.items()}

task_length = task['input_ids'].shape[1]

with torch.no_grad():
    gen_out = model.generate(
        [task],
        max_new_tokens=args.max_new_tokens,
        pad_token_id=eos[0],
        do_sample=False
    )

preds_full = torch.cat(gen_out, dim=1)
labels = collated['labels']
for i, (lab_tokens, pred_tokens) in enumerate(zip(labels, preds_full)):
    labels_mask = lab_tokens != -100
    lab_tokens = lab_tokens[labels_mask].tolist()
    lab_tokens = [t for t in lab_tokens if t != bos[0]]

    pred_tokens = pred_tokens.tolist()
    pred_tokens = [t for t in pred_tokens if t != bos[0]]
    
    ans_start_index_l = max(i for i, x in enumerate(lab_tokens) if x == ans[0])
    # ans_end_index_l = min(i for i, x in enumerate(lab_tokens) if x == eos[0])

    if ans[0] in pred_tokens:
        ans_start_index_p = max(i for i, x in enumerate(pred_tokens) if x in [ans[0], think[0]])
    else:
        ans_start_index_p = ans_start_index_l

    # if eos[0] in pred_tokens:
    #     ans_end_index_p = min(i for i, x in enumerate(pred_tokens) if x == eos[0])
    # else:
    #     ans_end_index_p = ans_end_index_l

    pred_cot_tokens = pred_tokens[:ans_start_index_p]
    lab_cot_tokens = lab_tokens[:ans_start_index_l]

    all_preds_cot.append(pred_cot_tokens)
    all_labels_cot.append(lab_cot_tokens)

    # pred_and_tokens = pred_tokens[ans_start_index_p+1:ans_end_index_p]
    pred_and_tokens = pred_tokens[ans_start_index_p+1:]
    # lab_ans_tokens = lab_tokens[ans_start_index_l+1:ans_end_index_l]
    lab_ans_tokens = lab_tokens[ans_start_index_l+1:]

    all_preds_ans.append(pred_and_tokens)
    all_labels_ans.append(lab_ans_tokens)

    all_preds.append(pred_tokens)
    all_labels.append(lab_tokens)

cot_correct = [p == l for p, l in zip(all_preds_cot, all_labels_cot)]
ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]

res = {'accuracy_cot': np.mean(cot_correct), 'accuracy_ans': np.mean(ans_correct)}
data = {"all_preds_cot": all_preds_cot,
        "all_labels_cot": all_labels_cot,
        "all_preds_ans": all_preds_ans,
        "all_labels_ans": all_labels_ans,
        "all_preds": all_preds,
        "all_labels": all_labels}

In [22]:
print("Pred:", tokenizer.decode(data["all_preds"][0]))
print("Lab: ", tokenizer.decode(data["all_labels"][0]))

Pred: 14-2=12<issue_start>12*2=24<issue_start>12+24=36<issue_start>36-3=33<issue_closed>33
Lab:  14/2=7<issue_start>15/3=5<issue_start>2*5=10<issue_start>14-10=4<issue_closed>4


In [18]:
print(all_preds_cot[0])

[33, 36, 29, 34, 45, 33, 34, 8, 33, 34, 26, 34, 45, 34, 36, 8, 33, 34, 27, 34, 36, 45, 35, 38, 8, 35, 38, 29, 35, 45, 35, 35]


In [14]:
print(all_labels_cot[0])

[33, 36, 31, 34, 45, 39, 8, 33, 37, 31, 35, 45, 37, 8, 34, 26, 37, 45, 33, 32, 8, 33, 36, 29, 33, 32, 45, 36]


In [15]:
tokenizer.decode(preds_full[0], skip_special_tokens=False)

'14-2=12<issue_start>12*2=24<issue_start>12+24=36<issue_start>36-3=33<issue_closed>33<|endoftext|>'

In [76]:
task_length

63

In [77]:
tokenizer.decode(pred_tokens)

'1.5*2=3<issue_start>1.5*2=3<issue_start>3*2.5=7.5<issue_start>3+3+2.5+7.5=15<issue_closed>15*15=225<issue_closed>225'

In [78]:
res

{'accuracy_cot': np.float64(0.0), 'accuracy_ans': np.float64(0.0)}

In [79]:
tokenizer.decode(data["all_labels_cot"][0])

'1.5*2=3<issue_start>3+2.5=5.5<issue_start>1.5+3+5.5=10'

In [80]:
tokenizer.decode(data["all_preds_cot"][0])

'1.5*2=3<issue_start>1.5*2=3<issue_start>3*2.5=7.5<issue_start>3+3+2.5+7.5=15<issue_closed>15*15=225'

In [81]:
tokenizer.decode(data["all_labels_ans"][0])

'10'

In [82]:
tokenizer.decode(data["all_preds_ans"][0])

'225'

In [84]:
ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]

In [85]:
ans_correct

[False]

In [11]:
I = 0

In [12]:
batch = valid_dataset.select(range(I, I+1))
collated = collate_fn(batch)
task = collated['segments'][0]
task = {k:v.to(device) for k,v in task.items()}

task_length = task['input_ids'].shape[1]

with torch.no_grad():
    gen_out = model.generate(
        [task],
        max_new_tokens=200,
        pad_token_id=eos[0],
        do_sample=False
    )

preds_full = torch.cat(gen_out, dim=1)
labels = collated['labels']

In [13]:
preds_full.shape, labels.shape

(torch.Size([1, 77]), torch.Size([1, 98]))

In [14]:
labels_masks = labels > 0
# preds_full = [p[m] for p, m in zip(preds_full, labels_masks)]
labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

In [17]:
labels_full[0].shape

torch.Size([32])

In [18]:
tokenizer.decode(labels_full[0])

'4-2=2<issue_start>2/.5=4<issue_start>12/4=3<issue_start>100*3=300<issue_closed>300'

In [ ]:
special_tokens = {ans[0], bos[0]}
for lab_tokens, pred_tokens in zip(labels_full, preds_full):
    ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

    pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
    lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

    cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]
    acc_cot.append(all(cot_correct))

    pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
    lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

    ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
    acc_ans.append(all(ans_correct))


res = {'accuracy_cot': float(np.mean(acc_cot)), 'accuracy_ans': float(np.mean(acc_ans))}

In [28]:
tokenizer.decode(task['input_ids'][0])

'<|endoftext|>John cuts his grass to 2 inches.  It grows .5 inches per month.  When it gets to 4 inches he cuts it back down to 2 inches.  It cost $100 to get his grass cut.  How much does he pay per year?<issue_start>'

In [31]:
preds_full[0].shape

torch.Size([77])

In [29]:
tokenizer.decode(preds_full[0])

'2*100=200<issue_start>200/5=40<issue_start>40*15=600<issue_closed>600/100=6<issue_closed>600/100=6<issue_start>6*2=12<issue_closed>12/2=6<issue_closed>6+6=12<issue_closed>12/2=6<issue_closed>'

In [30]:
labels[labels!=-100].shape

torch.Size([38])

In [33]:
tokenizer.decode(labels[labels!=-100])

'<|endoftext|>4-2=2<issue_start><|endoftext|>2/.5=4<issue_start><|endoftext|>12/4=3<issue_start><|endoftext|>100*3=300<issue_closed><|endoftext|>300<|endoftext|>'

## Find correctly predicted CoTs

In [13]:
bs = 8
dataset = valid_dataset

In [14]:
def collated_batch_to_device(collated):
    new_segments = []
    for segment in collated["segments"]:
        segment = {k: v.to(device) for k, v in segment.items()}
        new_segments.append(segment)

    collated["segments"] = new_segments
    
    collated["labels"] = collated["labels"].to(device)

    return collated

In [15]:
all_preds = []
all_labels = []
for start_ind in tqdm(range(0, len(dataset), bs)):
    batch = dataset.select(range(start_ind, min(len(dataset), start_ind + bs)))

    collated = collate_fn(batch)
    collated = collated_batch_to_device(collated)

    output = model(collated["segments"], collated["labels"], output_attentions=True)

    preds = output.logits.argmax(axis=-1)[:, :-1]
    labels = collated["labels"][:, 1:]

    all_preds.append(preds)
    all_labels.append(labels)

  0%|          | 0/63 [00:00<?, ?it/s]

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 8 but got size 4 for tensor number 62 in the list.

In [17]:
acc_cot = []
acc_ans = []
for preds, labels in zip(all_preds, all_labels):
    labels_masks = labels > 0
    preds_full = [p[m] for p, m in zip(preds, labels_masks)]
    labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

    special_tokens = {ans[0], bos[0]}
    acc_cot, acc_ans = [], []
    for lab_tokens, pred_tokens in zip(labels_full, preds_full):
        ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

        pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
        lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

        cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]
        acc_cot.append(all(cot_correct))

        pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
        lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

        ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
        acc_ans.append(all(ans_correct))

In [20]:
sum(acc_ans)

4

In [ ]:
all_preds.shape

## Validate model on full dataset

In [10]:
bs=1
dataset = valid_dataset

In [14]:
all_preds, all_labels = [], []
all_preds_cot, all_labels_cot = [], []
all_preds_ans, all_labels_ans = [], []

for start_ind in tqdm(range(0, len(dataset), bs)):
    batch = dataset.select(range(start_ind, min(len(dataset), start_ind + bs)))

    collated = collate_fn(batch)
    task = collated['segments'][0]
    task = {k:v.to(device) for k,v in task.items()}

    task_length = task['input_ids'].shape[1]

    with torch.no_grad():
        gen_out = model.generate(
            [task],
            max_new_tokens=args.max_new_tokens,
            pad_token_id=eos[0]
        )

    preds_full = torch.cat(gen_out, dim=1)
    labels = collated['labels']
    for i, (lab_tokens, pred_tokens) in enumerate(zip(labels, preds_full)):
        labels_mask = lab_tokens != -100
        lab_tokens = lab_tokens[labels_mask].tolist()
        lab_tokens = [t for t in lab_tokens if t != bos[0]]

        pred_tokens = pred_tokens.tolist()
        pred_tokens = [t for t in pred_tokens if t != bos[0]]
        
        ans_start_index_l = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

        if ans[0] in pred_tokens:
            ans_start_index_p = max(i for i, x in enumerate(pred_tokens) if x in [ans[0], think[0]])
        else:
            ans_start_index_p = ans_start_index_l


        pred_cot_tokens = pred_tokens[:ans_start_index_p]
        lab_cot_tokens = lab_tokens[:ans_start_index_l]

        all_preds_cot.append(pred_cot_tokens)
        all_labels_cot.append(lab_cot_tokens)

        pred_and_tokens = pred_tokens[ans_start_index_p+1:]
        lab_ans_tokens = lab_tokens[ans_start_index_l+1:]

        all_preds_ans.append(pred_and_tokens)
        all_labels_ans.append(lab_ans_tokens)

        all_preds.append(pred_tokens)
        all_labels.append(lab_tokens)

cot_correct = [p == l for p, l in zip(all_preds_cot, all_labels_cot)]
ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]

res = {'accuracy_cot': np.mean(cot_correct), 'accuracy_ans': np.mean(ans_correct)}
data = {"all_preds_cot": all_preds_cot,
        "all_labels_cot": all_labels_cot,
        "all_preds_ans": all_preds_ans,
        "all_labels_ans": all_labels_ans,
        "all_preds": all_preds,
        "all_labels": all_labels}

  0%|          | 0/500 [00:00<?, ?it/s]

In [15]:
print(res)

{'accuracy_cot': np.float64(0.006), 'accuracy_ans': np.float64(0.022)}
